In [1]:
import os
device = "cuda"
os.environ["DIGNN_ENV"] = device

import sys
sys.path.append('../..')

from DIGNN.data.Atoms import AtomsData
from DIGNN.data.utils import ase_db2AtomsData_list, update_basic_batch, update_cplt_graph
from DIGNN.utils.utils import AtomIndexMapper

from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

import time
import torch
import numpy as np
from ase.db import connect

e:\anaconda3\envs\tys\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0218 14:46:39.753000 15500 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
def generate_Graphs_from_db(file_path:str) -> list[AtomsData]:
    """
    basic_graphs: list of AtomsData, 只包含元素、坐标、力、能量和拓扑结构
    """
    ase_db = connect(file_path)
    basic_graphs = []
    start_time = time.time()
    
    for i,mol in enumerate(ase_db.select()):
        mol = mol.toatoms()
        atom_numbers = mol.get_atomic_numbers()
        pos = mol.positions
        force = mol.get_forces()
        ene = np.array([mol.get_potential_energy()])
        if ene[0] > 0: continue
        if np.abs(force).max() > 20: continue
        
        data = AtomsData(atom = torch.from_numpy(atom_numbers).long(),
                         pos=torch.from_numpy(pos).float())
        data.properties = ['force', 'ene']
        data.force = torch.from_numpy(force).float()
        data.ene = torch.from_numpy(ene).float()
        
        basic_graphs.append(data)

        if i % 1000 == 0:
            print(f'sample:{i},time_cost:{time.time() - start_time}')
            start_time = time.time()
    
    return basic_graphs

basic_data = generate_Graphs_from_db(r"E:\桌面临时文件\计算脚本测试\atoms.db")

In [ ]:
mapper = AtomIndexMapper(known_atom_nums=[5,67], device=device)

def atom_index_reset(basic_batch):
    for basic in basic_batch:
        basic.atom = mapper(basic.atom)
    return basic_batch

basic_data = atom_index_reset(basic_data)

In [ ]:
train_data, test_data = train_test_split(basic_data, test_size=0.2, random_state=42)

train_batch = DataLoader(train_data, batch_size=16, follow_batch=['atom'])
test_batch = DataLoader(test_data, batch_size=16, follow_batch=['atom'])


In [ ]:
pml_rcut = 2.0
pml_mnn = 12
iml_rcut = 4.0
iml_mnn = 16

train_basic_batch = update_basic_batch(train_batch,pml_rcut=pml_rcut, pml_mnn=pml_mnn, 
                                iml_rcut=iml_rcut, iml_mnn=iml_mnn, store_device='cpu')
test_basic_batch = update_basic_batch(test_batch, pml_rcut=pml_rcut, pml_mnn=pml_mnn, 
                                        iml_rcut=iml_rcut, iml_mnn=iml_mnn, store_device='cpu')


def get_cplt(basic):
    cplt = []
    for i,b in enumerate(basic):
        c = update_cplt_graph(b,
                            store_device=device, 
                            pos_grad=True, 
                            if_strip=True)
        cplt.append(c)
        
        if i % 100 == 0: print(f'cplt batch{i}')
    return cplt

In [ ]:
train_basic_batch1 = DataLoader(train_basic_batch, batch_size=1, shuffle=True,
                                num_workers=4, pin_memory=True, prefetch_factor=2)
test_basic_batch1 = DataLoader(test_basic_batch, batch_size=1, shuffle=False,
                                num_workers=4, pin_memory=True, prefetch_factor=2)


In [ ]:
from DIGNN.nn import models as dgm
from DIGNN.nn.utils import init_weights

def create_model():
    feature_dim = {'atom': 128, 'bond': 128, 'angle': 64, 'dihedral': 32}
    model = dgm.dignn.DIGNN(encoder=dgm.Encoder(num_species=100,
                                                atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml_rcut=pml_rcut+0.2,
                                                bondI_dim=feature_dim['bond'],
                                                iml_rcut=iml_rcut+0.2),
                    processor=dgm.GCN_Processor(atom_dim=feature_dim['atom'],
                                                bond_dim=feature_dim['bond'],
                                                ang_dim=feature_dim['angle'],
                                                dih_dim=feature_dim['dihedral'],
                                                pml=1,
                                                iml=4,
                                                residual=True,
                                                dropout=0.0,
                                                bondI_dim=feature_dim['bond'],
                                                init_nn_layer=0,
                                                ), 
                    decoder=dgm.Decoder(dim=[feature_dim['atom'],64,1], 
                                        reduce_method='sum', 
                                        dropout=0.0),
                    ).to(device)
    model.apply(init_weights)
    return model

model = create_model()

In [ ]:
loss_list = []

In [ ]:
from DIGNN.data.utils import batch_iterator

val = [[],[]]
def validate(model, validate_basic_batch):
    batch_num = len(validate_basic_batch)
    model.eval()

    mae_ene_epoch = 0
    mae_force_epoch = 0

    mae_criterion = torch.nn.L1Loss()
    
    for basic in validate_basic_batch:
        cplt = update_cplt_graph(basic.clone(), 
                                store_device=device, 
                                pos_grad=True,
                                if_strip=True)
        
        energy = model(cplt)
        force = -torch.autograd.grad(outputs=energy, 
                                    inputs=cplt.pos, 
                                    grad_outputs=torch.ones_like(energy),
                                    create_graph=False,
                                    retain_graph=True,
                                    )[0]
        mae_ene = mae_criterion(energy.view(-1,1), cplt.ene.view(-1,1))
        mae_force = mae_criterion(force.view(-1,3), cplt.force.view(-1,3))


        val[0].append(mae_ene.item())
        val[1].append(mae_force.item())

        mae_ene_epoch += mae_ene.item()
        mae_force_epoch += mae_force.item()

    print(f'validate mae ene: {mae_ene_epoch/batch_num}, force: {mae_force_epoch/batch_num}')

In [ ]:
def train(model, basic_batch, max_epoch=1000):
    runned_epoch = 0
    max_epoch -= runned_epoch

    batch_num = len(basic_batch)

    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer,
                                                    max_lr=1e-3,
                                                    total_steps=max_epoch*batch_num,
                                                    final_div_factor=1e+5,
                                                    )
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,
    #                                                                  T_0=int(max_epoch*batch_num*0.05),
    #                                                                  T_mult=2,
    #                                                                  eta_min=1e-8,
    #                                                                  last_epoch=-1)

    # check_point = torch.load('checkpoint_500.pth')
    # model.load_state_dict(check_point['model'])
    # optimizer.load_state_dict(check_point['optimizer'])
    # scheduler.load_state_dict(check_point['scheduler'])


    for epoch in range(max_epoch):
        epoch += runned_epoch
        model.train()
        loss_epoch = 0
        for i, basic in enumerate(basic_batch):
            cplt = update_cplt_graph(basic.clone(),
                                        store_device=device, 
                                        pos_grad=True, 
                                        if_strip=True)
            energy = model(cplt)
            force = -torch.autograd.grad(outputs=energy, 
                                    inputs=cplt.pos, 
                                    grad_outputs=torch.ones_like(energy),
                                    create_graph=True,
                                    retain_graph=True,
                                    )[0]

            loss_ene = criterion(energy.view(-1,1), cplt.ene.view(-1,1))
            loss_force = criterion(force.view(-1,3), cplt.force.view(-1,3))
            loss = 0.1 * loss_ene + 1.0 * loss_force
        
            loss.backward(retain_graph=False)
            optimizer.step()

            optimizer.zero_grad()
            scheduler.step()
                        
            loss_list.append(loss.item())
            loss_epoch += loss.item()

        
        if epoch % 1 == 0:
            print(f'epoch: {epoch+1}, loss: {loss_epoch/batch_num}')
        if (epoch+1) % 10 == 0:
            validate(model, test_basic_batch1)
        if (epoch+1) % 100 == 0:
            torch.save({'model': model.state_dict(),
                        'optimizer': optimizer.state_dict(),
                        'scheduler': scheduler.state_dict(),
                        'epoch': epoch,
                        }, f'checkpoint_{epoch+1}.pth')
    return model 

model = train(model, train_basic_batch,  max_epoch=100)


In [ ]:
def get_pred_target(batch_graphs):
    model.eval()
    
    pred_ene = []
    pred_force = []
    target_ene = []
    target_force = []
    
    for batch in update_cplt_data.batch_iterator(batch_graphs):
        cplt = update_cplt_graph(batch.clone(), 
                                device=device, 
                                pos_grad=True,
                                if_strip=True)
        energy = model(cplt)
        force = -torch.autograd.grad(energy, 
                                    cplt.pos, 
                                    grad_outputs=torch.ones_like(energy),
                                    retain_graph=True,
                                    create_graph=False)[0]

        pred_ene.append(energy.detach().cpu().view(-1,1).numpy())
        pred_force.append(force.detach().cpu().view(-1,3).numpy())
        
        target_ene.append(cplt.ene.detach().cpu().view(-1,1).numpy())
        target_force.append(batch.force.detach().cpu().view(-1,3).numpy())
    
    pred_ene = np.concatenate(pred_ene,axis=0)
    target_ene = np.concatenate(target_ene,axis=0)
    
    
    pred_force = np.concatenate(pred_force,axis=0)
    target_force = np.concatenate(target_force,axis=0)
    
    pred = [pred_ene.reshape(-1,1), pred_force.reshape(-1,3)]
    target = [target_ene.reshape(-1,1), target_force.reshape(-1,3)]
    return pred, target

test_i = 1
test_lst = [basic_batch, test_basic_batch]
pred, target = get_pred_target(test_lst[test_i])


test_atom_num = [train_data, test_data]
atoms_num = []
for d in test_atom_num[test_i]:
    atoms_num.append(d.atom.shape[0])

atoms_num = np.array(atoms_num)

In [ ]:
opt = 0
atoms_num = atoms_num

if opt == 0 : xrange = (-300,50)
if opt == 1 : xrange = (-100,100)
if opt == 2 : xrange = (-15,-1)

plt.figure(dpi=300,figsize=(7,5.3))


if opt == 0:
    plot_comparison(target[opt],pred[opt],atom_num=atoms_num, *xrange)
if opt == 1:
    plot_comparison(target[opt],pred[opt],atom_num=None, *xrange)
if opt == 2:
    plot_comparison((target[0]/atoms_num).reshape(-1,1),(pred[0]/atoms_num).reshape(-1,1), 
                    atom_num=None, *xrange)
    
plt.plot([*xrange], [*xrange], color='r', linestyle='-',linewidth=0.5)  # 添加 y=x 的线

plt.xlim(*xrange)
plt.ylim(*xrange)
plt.xticks()
plt.yticks()
#plt.grid(False)
plt.xlabel('target')
plt.ylabel('pred')